In [1]:
from cresowlve.utils import read_json

model_results_path = "../experiments/outputs/gemini-3-flash-preview/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-3-flash-preview_20260302_151010.json"
benchmark_path = "../experiments/data/task/chgk_en_benchmark.json"
culture_lang_path = "../experiments/data/task/chgk_benchmark_culture_lang.json"
model_results = read_json(model_results_path)
benchmark = read_json(benchmark_path)
culture_lang_results = read_json(culture_lang_path)
reasoning_type_results = read_json("../experiments/data/task/chgk_benchmark_reasoning_types.json")
factual_ids = [s["id"] for s in reasoning_type_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "factual"]
creative_ids = [s["id"] for s in reasoning_type_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "creative"]
culture_lang_results["data"] = [s for s in culture_lang_results["data"] if s["id"] in creative_ids]

In [2]:
from collections import Counter
culture_langs = Counter([cl for s in culture_lang_results["data"] for cl in s["culture_langs"]])

culture_langs

Counter({'english': 1186,
         'russian': 796,
         'french': 210,
         'german': 160,
         'greek': 134,
         'italian': 119,
         'latin': 85,
         'spanish': 75,
         'american': 75,
         'japanese': 54,
         'polish': 34,
         'arabic': 26,
         'dutch': 24,
         'roman': 23,
         'hebrew': 21,
         'swedish': 21,
         'chinese': 21,
         'ukrainian': 19,
         'norwegian': 17,
         'danish': 16,
         'indian': 16,
         'portuguese': 14,
         'scottish': 13,
         'czech': 13,
         'egyptian': 12,
         'georgian': 10,
         'turkish': 9,
         'european': 9,
         'persian': 9,
         'swiss': 9,
         'irish': 9,
         'biblical': 8,
         'finnish': 8,
         'brazilian': 8,
         'hungarian': 7,
         'christian': 7,
         'christianity': 7,
         'serbian': 6,
         'scandinavian': 6,
         'canadian': 6,
         'romanian': 6,
         'bul

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import squarify
 
# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
culture_counts = {c: val for c, val in culture_langs.items() if val > 10}
culture_counts["other"] = sum(val for val in culture_langs.values() if val <= 10)

# ─────────────────────────────────────────────────────────────────────────────
 
total = len(culture_lang_results["data"]) # sum(culture_counts.values())
df = pd.DataFrame({
    "name":  list(culture_counts.keys()),
    "value": list(culture_counts.values()),
})
df["pct"] = df["value"] / total * 100
 
# ── Light pastel palette ──────────────────────────────────────────────────────
N      = len(df)
hues   = np.linspace(0, 1, N, endpoint=False)
# colors = [mcolors.hsv_to_rgb([h, 0.30, 0.96]) for h in hues]
# colors = [plt.cm.Pastel2(i / N) for i in range(N)]
colors = [plt.cm.Set3(i / N) for i in range(N)]
 
# ── Labels: concept name + count + percentage ─────────────────────────────────
labels = [
    f"{name}\n{value} ({pct:.1f}%)"
    for name, value, pct in zip(df["name"], df["value"], df["pct"])
]
 
# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_axis_off()
 
squarify.plot(
    sizes=np.sqrt(df["value"]),
    label=labels,
    color=colors,
    text_kwargs={"color": "#333333", "fontsize": 16, "fontweight": "bold"},
    pad=True,
    ax=ax,
)
 
# plt.savefig("../figures/culture_langs_dist.pdf",
#             dpi=360, bbox_inches="tight", facecolor="white")
plt.savefig("../figures/culture_langs_dist.png",
            dpi=360, bbox_inches="tight", facecolor="white")
plt.close()
print("Done")

Done


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import matplotlib.colors as mcolors

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
num_langs_dist = Counter([len(s["culture_langs"]) for s in culture_lang_results["data"] if "culture_langs" in s])
# ─────────────────────────────────────────────────────────────────────────────

df = pd.DataFrame({
    "langs": list(num_langs_dist.keys()),
    "count":   list(num_langs_dist.values()),
})
df = df.sort_values("langs").reset_index(drop=True)
df["langs"] = df["langs"].astype(str)

# ── Light pastel palette (one color per bar) ──────────────────────────────────
N      = len(df)
hues   = np.linspace(0, 1, N, endpoint=False)
colors = [mcolors.hsv_to_rgb([h, 0.30, 0.96]) for h in hues]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6), facecolor="white")
ax.set_facecolor("white")

sns.barplot(data=df, x="langs", y="count", palette=colors, ax=ax)

# Count + percentage labels on top of each bar
total = df["count"].sum()
for bar in ax.patches:
    count = int(bar.get_height())
    pct = count / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.003,
        f"{count}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=20, fontweight="bold", color="#444444"
    )

ax.set_xlabel("Number of cultures/languages", fontsize=24, color="#444444")
ax.set_ylabel("Number of questions", fontsize=24, color="#444444")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(colors="#666666")
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)

plt.tight_layout()
plt.savefig("../figures/num_langs_dist.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_93881/549543094.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x="langs", y="count", palette=colors, ax=ax)


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
models = [
    ("Gemini-3.1-Pro (medium)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-3.1-pro-preview_20260310_105704.json"),
    ("Gemini-3.1-Pro (high)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-3.1-pro-preview_20260312_204451.json"),
    ("GPT-4.1", "../experiments/outputs/gpt-4.1/chgk_en_benchmark_eval_cot_answer_en_s0_gpt-4.1_20260228_123352.json"),
    ("Qwen3-235B-A22B-Thinking", "../experiments/outputs/qwen3-235b-a22b-thinking-2507/chgk_en_benchmark_eval_reasoning_model_en_s0_qwen3-235b-a22b-thinking-2507_20260312_203137.json"),
    ("DeepSeek-V3.2", "../experiments/outputs/deepseek-v3.2/chgk_en_benchmark_eval_reasoning_model_en_s0_deepseek-v3.2_20260315_133331.json"),
    ("GPT-5.4 (medium)", "../experiments/outputs/gpt-5.4/chgk_en_benchmark_eval_reasoning_model_en_s0_gpt-5.4_20260310_110106.json"),
]
results = {}
cultures = [culture for culture, counts in culture_langs.items() if counts > 30]
cultures = sorted(cultures, key=lambda c: culture_langs[c], reverse=True)

for model, model_path in models:
    model_results = read_json(model_path)
    model_accs = []
    for culture in cultures:
        culture_sample_ids = [s["id"] for s in culture_lang_results["data"] if culture in s.get("culture_langs", [])]
        culture_samples = [s for s in model_results["data"] if s["id"] in culture_sample_ids]
        culture_acc = np.mean([int(s.get("gpt-4o_judge_match", False)) for s in culture_samples])
        model_accs.append(culture_acc)
    results[model] = model_accs
# ─────────────────────────────────────────────────────────────────────────────

df = pd.DataFrame(results, index=[f"{culture} ({culture_langs[culture]})" for culture in cultures]).T

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, len(results)), facecolor="white")
cmap = sns.light_palette("#7a5c3a", as_cmap=True)  # cream → warm brown

sns.heatmap(
    df,
    ax=ax,
    cmap=cmap,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 9},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Accuracy", "shrink": 0.6},
    vmin=0, vmax=1,
)

ax.set_xlabel("Culture", fontsize=12, color="#444444")
ax.set_ylabel("Model", fontsize=12, color="#444444")
ax.tick_params(axis="x", labelsize=11, colors="#444444", rotation=45)
ax.tick_params(axis="y", labelsize=9,  colors="#444444", rotation=0)

plt.tight_layout()
plt.savefig("../figures/culture-results.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
models = [
    ("Gemini-3.1-Pro (medium)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gemini-3.1-pro-preview_20260310_160621.json"),
    ("Gemini-3.1-Pro (high)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gemini-3.1-pro-preview_20260313_232947.json"),
    ("GPT-4.1", "../experiments/outputs/gpt-4.1/chgk_ru_benchmark_eval_cot_answer_ru_en_s0_gpt-4.1_20260228_160027.json"),
    ("Qwen3-235B-A22B-Thinking", "../experiments/outputs/qwen3-235b-a22b-thinking-2507/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_qwen3-235b-a22b-thinking-2507_20260314_102839.json"),
    ("DeepSeek-V3.2", "../experiments/outputs/deepseek-v3.2/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_deepseek-v3.2_20260315_165350.json"),
    ("GPT-5.4 (medium)", "../experiments/outputs/gpt-5.4/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gpt-5.4_20260310_132133.json"),
]
results = {}
cultures = [culture for culture, counts in culture_langs.items() if counts > 30]
cultures = sorted(cultures, key=lambda c: culture_langs[c], reverse=True)

for model, model_path in models:
    model_results = read_json(model_path)
    model_accs = []
    for culture in cultures:
        culture_sample_ids = [s["id"] for s in culture_lang_results["data"] if culture in s.get("culture_langs", [])]
        culture_samples = [s for s in model_results["data"] if s["id"] in culture_sample_ids]
        culture_acc = np.mean([int(s.get("gpt-4o_judge_match", False)) for s in culture_samples])
        model_accs.append(culture_acc)
    results[model] = model_accs
# ─────────────────────────────────────────────────────────────────────────────

df = pd.DataFrame(results, index=[f"{culture} ({culture_langs[culture]})" for culture in cultures]).T

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, len(results)), facecolor="white")
cmap = sns.light_palette("#7a5c3a", as_cmap=True)  # cream → warm brown

sns.heatmap(
    df,
    ax=ax,
    cmap=cmap,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 9},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Accuracy", "shrink": 0.6},
    vmin=0, vmax=1,
)

ax.set_xlabel("Culture", fontsize=12, color="#444444")
ax.set_ylabel("Model", fontsize=12, color="#444444")
ax.tick_params(axis="x", labelsize=11, colors="#444444", rotation=45)
ax.tick_params(axis="y", labelsize=9,  colors="#444444", rotation=0)

plt.tight_layout()
plt.savefig("../figures/culture-results-ru.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()